In [2]:
import pandas as pd
import numpy as np

In [5]:
from glaciation_time_estimator.auxiliary_func.chunking_data import ChunkLoader
from glaciation_time_estimator.auxiliary_func.config_reader import read_config

In [19]:
years=[y for y in range(2007,2011)]
config = read_config("/wolke_scratch/dnikolo/Glaciation_time_estimator/configs/Validation_DARDAR/02_deg/2008_tracking/01_01.yaml")


In [20]:
loader = ChunkLoader(years,config,load_init_chunk=False)

Cloud_fps: {2007: '/nfs/ch4/wolke_scratch/dnikolo/GTE_Validation/DARDAR/Bulk_02/Final_results/2007_all.parquet', 2008: '/nfs/ch4/wolke_scratch/dnikolo/GTE_Validation/DARDAR/Bulk_02/Final_results/2008_all.parquet', 2009: '/nfs/ch4/wolke_scratch/dnikolo/GTE_Validation/DARDAR/Bulk_02/Final_results/2009_all.parquet', 2010: '/nfs/ch4/wolke_scratch/dnikolo/GTE_Validation/DARDAR/Bulk_02/Final_results/2010_all.parquet'}


In [21]:
def extract_relative_ice_fraction(
    df,
    hist_col="ice_frac_hist",
    peak_col="Glac_peak_ind",
    rel_range=(-8, 9),
    eps=1e-4,
    demean_within_cloud=False,
):
    rel_inds = np.arange(rel_range[0], rel_range[1])

    rows = []
    cloud_ids = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting relative IF"):
        hist = row[hist_col]
        peak_ind = row[peak_col]

        if hist is None or pd.isna(peak_ind):
            continue

        x = np.asarray(hist, dtype=float)
        peak_ind = int(peak_ind)

        valid = (
            np.isfinite(x)
            & (x >= 0 - eps)
            & (x <= 1 + eps)
        )

        x = x.copy()
        x[~valid] = np.nan

        if np.isfinite(x).sum() < 4:
            continue

        if demean_within_cloud:
            x = x - np.nanmean(x)

        vals = []

        for rel_ind in rel_inds:
            abs_ind = peak_ind + rel_ind

            if abs_ind < 0 or abs_ind >= len(x):
                vals.append(np.nan)
            else:
                vals.append(x[abs_ind])

        rows.append(vals)
        cloud_ids.append(idx)

    return pd.DataFrame(rows, index=cloud_ids, columns=rel_inds)

In [22]:
def correlation_with_peak(rel_df, peak_rel_ind=0, min_samples=30):
    """
    Correlate every relative-time column with the peak column.
    """
    peak = rel_df[peak_rel_ind]

    results = []

    for rel_ind in rel_df.columns:
        y = rel_df[rel_ind]

        mask = np.isfinite(peak) & np.isfinite(y)

        n = mask.sum()

        if n < min_samples:
            corr = np.nan
        else:
            corr = np.corrcoef(peak[mask], y[mask])[0, 1]

        results.append({
            "rel_ind": rel_ind,
            "rel_time_min": rel_ind * 15,
            "corr_with_peak": corr,
            "n_clouds": n,
        })

    return pd.DataFrame(results)

In [23]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


def pearson_corr_manual(x, y, min_samples=30):
    mask = np.isfinite(x) & np.isfinite(y)

    if mask.sum() < min_samples:
        return np.nan

    x = np.asarray(x[mask], dtype=float)
    y = np.asarray(y[mask], dtype=float)

    x = x - np.mean(x)
    y = y - np.mean(y)

    denom = np.sqrt(np.sum(x**2) * np.sum(y**2))

    if denom == 0:
        return np.nan

    return np.sum(x * y) / denom


def correlation_with_peak_ci(
    rel_df,
    peak_rel_ind=0,
    min_samples=30,
    n_boot=500,
    ci=(2.5, 97.5),
    random_state=0,
):
    """
    Correlation of each relative-time column with the peak column,
    with cloud-level bootstrap confidence intervals.

    rel_df rows = clouds
    rel_df columns = relative time indices
    """

    rng = np.random.default_rng(random_state)

    rel_cols = np.asarray(rel_df.columns)
    values = rel_df.to_numpy(dtype=float)

    if peak_rel_ind not in rel_df.columns:
        raise ValueError(f"peak_rel_ind={peak_rel_ind} not found in rel_df columns")

    peak_col_pos = np.where(rel_cols == peak_rel_ind)[0][0]

    n_clouds_total = values.shape[0]
    n_lags = values.shape[1]

    # Main estimate
    main_corrs = np.full(n_lags, np.nan)
    n_clouds = np.full(n_lags, 0, dtype=int)

    peak = values[:, peak_col_pos]

    for j in range(n_lags):
        y = values[:, j]
        mask = np.isfinite(peak) & np.isfinite(y)
        n_clouds[j] = mask.sum()
        main_corrs[j] = pearson_corr_manual(peak, y, min_samples=min_samples)

    # Bootstrap estimates
    boot_corrs = np.full((n_boot, n_lags), np.nan)

    for b in tqdm(range(n_boot), desc="Bootstrap correlations"):
        inds = rng.integers(0, n_clouds_total, size=n_clouds_total)

        boot_values = values[inds, :]
        boot_peak = boot_values[:, peak_col_pos]

        for j in range(n_lags):
            y = boot_values[:, j]
            boot_corrs[b, j] = pearson_corr_manual(
                boot_peak,
                y,
                min_samples=min_samples,
            )

    ci_low = np.nanpercentile(boot_corrs, ci[0], axis=0)
    ci_high = np.nanpercentile(boot_corrs, ci[1], axis=0)

    out = pd.DataFrame({
        "rel_ind": rel_cols,
        "rel_time_min": rel_cols * 15,
        "corr_with_peak": main_corrs,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "n_clouds": n_clouds,
    })

    return out

In [24]:
def return_glac_df(a):
    return a[1]

In [25]:
glac_dfs = loader.execute_analysis(return_glac_df,load_clouds=False)

Loading years: 100%|██████████| 4/4 [00:10<00:00,  2.59s/year]


In [29]:
combined_glac_df = pd.concat(glac_dfs,ignore_index=True)

In [30]:
rel_if_anom = extract_relative_ice_fraction(
    combined_glac_df,
    rel_range=(-8, 9),
    demean_within_cloud=True,
)

corr_peak_ci = correlation_with_peak_ci(
    rel_if_anom,
    peak_rel_ind=0,
    min_samples=100,
    n_boot=100,
    random_state=0,
)


Extracting relative IF:   0%|          | 0/347634 [00:00<?, ?it/s]

Bootstrap correlations:   0%|          | 0/100 [00:00<?, ?it/s]

In [32]:
corr_peak_ci.to_parquet("/wolke_scratch/dnikolo/MSGI/plotting/IF_autocorrelation_values.parquet")